# Section 1. Install deps


In [2]:
!pip install transformers torch accelerate

# Section 2. Test cases


In [1]:
test_examples = [
  {
    "case_id": "case_001",
    "description": "Простий кейс (Happy Path)",
    "input": "Компанія Apple інвестувала 2 мільярди доларів у розвиток штучного інтелекту 15 травня 2024 року в США.",
    "expected_behavior": "Extract all fields correctly: Apple, investment, 2000000000, USD, 2024-05-15, USA. No repair needed."
  },
  {
    "case_id": "case_002",
    "description": "Missing required field",
    "input": "Стало відомо про інвестицію у розмірі 100 млн євро, яку виділили для стартапу минулого тижня.",
    "expected_behavior": "Validator should flag missing 'company' and 'location'; trigger repair or fail."
  },
  {
    "case_id": "case_003",
    "description": "Ambiguous entity",
    "input": "Amazon купує частину бізнесу своєї дочірньої компанії Amazon Robotics.",
    "expected_behavior": "Extract main company 'Amazon' (mentioned first) and event 'acquisition'; handle naming ambiguity."
  },
  {
    "case_id": "case_004",
    "description": "Relative date",
    "input": "Вчора корпорація Microsoft підписала партнерську угоду на суму 300 млн доларів.",
    "expected_behavior": "Normalize 'Вчора' to a specific YYYY-MM-DD date based on current metadata."
  },
  {
    "case_id": "case_005",
    "description": "Potential Hallucination",
    "input": "Tesla оголосила про запуск нового заводу в Німеччині. Аналітики вважають, що це коштуватиме мільярди.",
    "expected_behavior": "Amount should be 'null' because 'мільярди' is a vague estimate, not a specific number. Prevent model from inventing a digit."
  },
  {
    "case_id": "case_006",
    "description": "Noisy text / Typos",
    "input": "Googlr інвестує 500млн $ у Францію (Париж).",
    "expected_behavior": "Fix typo 'Googlr' -> 'Google'; extract '500000000', 'USD', 'France'."
  },
  {
    "case_id": "case_007",
    "description": "Fallback scenario",
    "input": "У новому звіті йдеться про велику угоду в агросекторі між двома фермерськими господарствами.",
    "expected_behavior": "Extractor fails to find specific names/amounts; system returns empty schema or marks as 'other' for fallback processing."
  },
  {
    "case_id": "case_008",
    "description": "Reviewer rejection",
    "input": "Акції Apple зросли на 5.5% після презентації в Купертіно.",
    "expected_behavior": "Reviewer/Validator should reject this as 'investment' because stock price change is not a corporate investment event."
  },
  {
    "case_id": "case_009",
    "description": "Repair loop helps",
    "input": "Meta виділяє 1 мільярд євро на безпеку.",
    "expected_behavior": "Initial output might miss 'location'. Repair prompt asks for location; model finds context or sets to null/location from metadata."
  },
  {
    "case_id": "case_010",
    "description": "Repair fails / Manual review",
    "input": "Компанія X-Corp (колишня Twitter) уклала угоду з приватною фірмою в ОАЕ.",
    "expected_behavior": "Model confuses X-Corp with SpaceX or fails to extract amount. After 2 repair attempts, flag for manual review."
  }
]

# Section 3. Agent role definitions


In [2]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="cpu"
)
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [38]:
import re

def ask_agent(prompt, system_message):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    outputs = pipe(text, max_new_tokens=512, do_sample=False, return_full_text=False)
    return outputs[0]["generated_text"].strip()

def safe_parse(text):
    text = re.sub(r"```json\s*", "", text)
    text = re.sub(r"```", "", text).strip()

    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        raise ValueError(f"No JSON found in: {text}")

    return json.loads(match.group(0))

# --- Agent 1 — Triager ---
def run_triager(user_input):
    system_msg = """You are Agent 1 (Triager). Your task: to classify the incoming news text.
    You MUST return ONLY valid JSON with the following fields:
    - task_type ("support_news_extraction")
    - route (schema name: "news_schema")
    - expected_fields (list of fields to extract: ["company", "event_type", "amount", "currency", "date", "location"])
    - difficulty (low/medium/high)
    - notes (short comment)

    Output format example:
    {
      "task_type": "support_news_extraction",
      "route": "news_schema",
      "expected_fields": ["company", "event_type", "amount", "currency", "date", "location"],
      "difficulty": "medium",
      "notes": "contains amount, currency and company name; date is relative"
    }
    Don't do the extraction by yourself."""

    response = ask_agent(user_input, system_msg)
    return safe_parse(response)

# --- Agent 2 — Extractor ---
def run_extractor(user_input, triage_data):
    system_msg = f"""You are Agent 2 (Extractor). Your task: extract data from the text according to the schema.
    Schema from Triager: {triage_data['expected_fields']}.
    Rules:
    - Return ONLY valid JSON.
    - Set null if there is no data (fields are missing).
    - Add a 'confidence_note' field with an explanation of the confidence.
    - Don't make up data!
    - Use YYYY-MM-DD for dates if possible.
    - Amount must be a number.
    - Extract only the core brand name.
    - event_type must be one of ["acquisition", "investment", "partnership", "launch", "sanctions", "other"].
    - Normalize currency (e.g., "доларів" → USD, "€" -> EUR, "$" -> USD).
    - Choose the main company which mentioned first if multiple are mentioned.

    Output schema:
    {{
      "company": string,
      "event_type": lowercase string,
      "amount": number or null,
      "currency": string or null,
      "date": string or null,
      "location": string or null,
      "confidence_note": string
    }}"""

    response = ask_agent(user_input, system_msg)
    return safe_parse(response)

# --- Agent 3 — Reviewer ---
def run_reviewer(user_input, extraction_data):
    system_msg = """You are Agent 3 (Reviewer). Check the Extractor output: compare extraction with input text; check consistency;
    find out if there are missing/hallucinated/contradictory fields.
    Return JSON with fields:
    - verdict (accept / repair_needed / fallback_needed)
    - valid_json (boolean)
    - schema_ok (boolean)
    - consistency_ok (boolean)
    - issues (list of objects with field and problem)
    - recommended_action (what to do next)

    Output format example:
    {
      "verdict": "repair_needed",
      "valid_json": true,
      "schema_ok": true,
      "consistency_ok": false,
      "issues": [
        {
        "field": "date",
        "problem": "date not normalized"
        }
      ],
      "recommended_action": "run_repair_with_date_normalization"
    }"""

    prompt = f"Input text: {user_input}\nExtraction output: {json.dumps(extraction_data)}"
    response = ask_agent(prompt, system_msg)
    return safe_parse(response)



# Section 4. Delegation rules


In [26]:
def delegate_task(text):
    print(f"--- Start Triager ---")
    triage = run_triager(text)
    print(triage)

    print(f"--- Routing to Extractor ---")
    extraction = run_extractor(text, triage.get("expected_fields"))
    print(extraction)

    print(f"--- Routing to Reviewer ---")
    attempts = 0
    while attempts < 2:
        review = run_reviewer(text, extraction)
        print(review)

        if review["verdict"] == "accept":
            return {"status": "success", "data": extraction}

        if review["verdict"] == "repair_needed":
            print(f"Repair attempt {attempts + 1}")
            extraction = run_fallback_repair(text, extraction, review["issues"])
            attempts += 1
        else:
            break

    return {"status": "manual_review", "last_data": extraction, "reason": "Repair failed"}

# Section 5. Fallback logic


In [ ]:
def run_fallback_repair(user_input, previous_extraction, issues):
    formatted_issues = "\n".join([f"- Field '{i['field']}': {i['problem']}" for i in issues])

    system_msg = f"""You are Extractor in REPAIR mode.
    Your task is to fix the previous extraction based on the Reviewer's comments.

    ORIGINAL TEXT: {user_input}
    PREVIOUS VARIANT: {json.dumps(previous_extraction, ensure_ascii=False)}

    ERRORS FOUND:
    {formatted_issues}

    RULES:
    1. Return ONLY corrected, complete, valid JSON.
    2. Fix the errors listed above based on the original text.
    3. If the Reviewer pointed out a hallucination, replace this field with the correct value or null.
    4. Do not add any text and fields other than JSON.
    5. Keep all other uncommented fields as they are.

    Output format example:
    {
      "status": "failed",
      "reason": "required field amount is missing",
      "corrected_fields": {
          {
            "amount": 1800000000,
          }
      },
      "needs_manual_review": true
    }"""

    repair_prompt = "Provide the corrected JSON according to the instructions."
    response = ask_agent(repair_prompt, system_msg)

    try:
        fixed_json = json.loads(response)
        return fixed_json
    except json.JSONDecodeError:
        print("CRITICAL: Repair returned invalid JSON format.")
        return previous_extraction


# Section 6. Reviewer consistency checks


In [5]:
def consistency_reviewer(user_input, extraction_data):
    system_msg = """You are Reviewer. Check the Extractor output: compare extraction with input text; check consistency;
    find out if there are missing/hallucinated/contradictory fields.
    Return JSON with fields:
    - verdict (accept / repair_needed / fallback_needed)
    - valid_json (boolean)
    - schema_ok (boolean)
    - consistency_ok (boolean)
    - issues (list of objects with field and problem)
    - recommended_action (what to do next)

    Output format example:
    {
      "verdict": "repair_needed",
      "valid_json": true,
      "schema_ok": true,
      "consistency_ok": false,
      "issues": [
        {
        "field": "date",
        "problem": "date not normalized"
        }
      ],
      "recommended_action": "run_repair_with_date_normalization"
    }
    """

    prompt = f"Input text: {user_input}\nExtraction output: {json.dumps(extraction_data)}"
    response = ask_agent(prompt, system_msg)
    return safe_parse(response)

# Section 7. Single-agent baseline


In [6]:
def run_single_agent_baseline(input_text):

    system_msg = """You are a precise information extraction system.
    Rules:
    - Respond ONLY with valid JSON.
    - If missing → null
    - Use YYYY-MM-DD for dates if possible
    - Amount must be a number
    - Extract only the core brand name
    - event_type must be one of ["acquisition", "investment", "partnership", "launch", "sanctions", "other"]
    - Normalize currency (e.g., "доларів" → USD, "€" -> EUR, "$" -> USD)
    - Choose the main company which mentioned first if multiple are mentioned

    Output schema:
    {{
      "company": string,
      "event_type": lowercase string,
      "amount": number or null,
      "currency": string or null,
      "date": string or null,
      "location": string or null
    }}"""

    try:
        response = ask_agent(input_text, system_msg)

        baseline_extraction = json.loads(response)
        return baseline_extraction
    except Exception as e:
        return {
            "error": "Failed to extract or parse JSON",
            "raw_response": response if 'response' in locals() else str(e)
        }

# Section 8. Multi-agent crew workflow


In [23]:
def crew_workflow(text):
    print(f"--- Start Triager ---")
    triage = run_triager(text)

    print(f"--- Routing to Extractor ---")
    extraction = run_extractor(text, triage)

    print(f"--- Routing to Reviewer ---")
    attempts = 0
    repair = {}

    while attempts < 2:
        review = run_reviewer(text, extraction)

        if review["verdict"] == "accept":
            return triage, extraction, review, repair

        if review["verdict"] == "repair_needed":
            print(f"Repair attempt {attempts + 1}")
            repair = run_fallback_repair(text, extraction, review["issues"])
            extraction = repair
            attempts += 1
        else:
            break

    return triage, extraction, review, repair

# Section 9. Run 10 test cases


In [18]:
def execute_single_test_suite(test_examples):
    all_results = []

    for example in test_examples:
        case_id = example["case_id"]
        input_text = example["input"]

        print(f"\n{'='*30}\nEXECUTING (Single): {case_id}\n{'='*30}")

        try:
            single = run_single_agent_baseline(input_text)
            check = consistency_reviewer(input_text, single)
            res = {
                "case_id": case_id,
                "input": input_text,
                "output": single,
                "review": check,
                "status": "success" if "error" not in single else "failed"
            }
            print(res)

            all_results.append(res)

        except Exception as e:
            print(f"Error in {case_id}: {e}")
            res = {
                "case_id": case_id,
                "status": "failed",
                "error": str(e)
            }
            print(res)
            all_results.append(res)

    return all_results

In [24]:
def execute_test_suite(test_examples):
    all_results = []

    for example in test_examples:
        case_id = example["case_id"]
        input_text = example["input"]

        print(f"\n{'='*30}\nEXECUTING (Crew): {case_id}\n{'='*30}")

        try:
            triage, extraction, review, repair = crew_workflow(input_text)

            is_repaired = bool(repair)
            if is_repaired == False:
                repair = extraction
            res = {
                "case_id": case_id,
                "input": input_text,
                "triager_output": {
                    "route": triage.get("route"),
                    "difficulty": triage.get("difficulty")
                },
                "extractor_output": {
                    "company": extraction.get("company"),
                    "event_type": extraction.get("event_type"),
                    "amount": extraction.get("amount"),
                    "currency": extraction.get("currency"),
                    "date": extraction.get("date"),
                    "location": extraction.get("location")
                },
                "reviewer_output": {
                    "verdict": review.get("verdict"),
                    "issues": review.get("issues")
                },
                "fallback_triggered": "true" if is_repaired else "false",
                "fallback_output":{
                    "date_normalized": "null",
                    "needs_manual_review": "true"
                },
                "final_output":{
                    "company": repair.get("company"),
                    "event_type": repair.get("event_type"),
                    "amount": repair.get("amount"),
                    "currency": repair.get("currency"),
                    "date": repair.get("date"),
                    "location": repair.get("location"),
                    "date_normalized": "null",
                    "needs_manual_review": "true" if is_repaired else "false"
                },
                "status": "accepted_after_repair" if is_repaired else "accepted_immediately"
            }
            print(res)
            all_results.append(res)

        except Exception as e:
            print(f"Error in {case_id}: {e}")
            res = {
                "case_id": case_id,
                "status": "failed",
                "error": str(e)
            }
            print(res)
            all_results.append(res)

    return all_results

In [19]:
single_results = execute_single_test_suite(test_examples)

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



EXECUTING (Single): case_001


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'case_id': 'case_001', 'input': 'Компанія Apple інвестувала 2 мільярди доларів у розвиток штучного інтелекту 15 травня 2024 року в США.', 'output': {'company': 'Apple', 'event_type': 'investment', 'amount': 2000000000, 'currency': 'USD', 'date': '2024-03-15', 'location': 'США'}, 'review': {'verdict': 'accept', 'valid_json': True, 'schema_ok': True, 'consistency_ok': True, 'issues': [], 'recommended_action': ''}, 'status': 'success'}

EXECUTING (Single): case_002


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

{'case_id': 'case_008', 'input': 'Акції Apple зросли на 5.5% після презентації в Купертіно.', 'output': {'company': 'Apple', 'event_type': 'launch', 'amount': 5.5, 'currency': '%', 'date': '2023-10-04', 'location': 'Купертіно'}, 'review': {'verdict': 'accept', 'valid_json': True, 'schema_ok': True, 'consistency_ok': True, 'issues': [], 'recommended_action': ''}, 'status': 'success'}

EXECUTING (Single): case_009


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'case_id': 'case_009', 'input': 'Meta виділяє 1 мільярд євро на безпеку.', 'output': {'company': 'Meta', 'event_type': 'investment', 'amount': 1000000000, 'currency': 'EUR', 'date': None, 'location': None}, 'review': {'verdict': 'accept', 'valid_json': True, 'schema_ok': True, 'consistency_ok': True, 'issues': [], 'recommended_action': ''}, 'status': 'success'}

EXECUTING (Single): case_010


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'case_id': 'case_010', 'input': 'Компанія X-Corp (колишня Twitter) уклала угоду з приватною фірмою в ОАЕ.', 'output': {'company': 'X-Corp', 'event_type': 'acquisition', 'amount': None, 'currency': None, 'date': 'YYYY-MM-DD', 'location': 'ОАЕ'}, 'review': {'verdict': 'accept', 'valid_json': True, 'schema_ok': True, 'consistency_ok': True, 'issues': [], 'recommended_action': ''}, 'status': 'success'}


In [25]:
all_results = execute_test_suite(test_examples)

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



EXECUTING (Crew): case_001
--- Start Triager ---


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- Routing to Extractor ---


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

{'case_id': 'case_009', 'input': 'Meta виділяє 1 мільярд євро на безпеку.', 'triager_output': {'route': 'news_schema', 'difficulty': 'high'}, 'extractor_output': {'company': 'Meta', 'event_type': 'investment', 'amount': 1000000000, 'currency': 'EUR', 'date': None, 'location': None}, 'reviewer_output': {'verdict': 'accept', 'issues': []}, 'fallback_triggered': 'false', 'fallback_output': {'date_normalized': 'null', 'needs_manual_review': 'true'}, 'final_output': {'company': 'Meta', 'event_type': 'investment', 'amount': 1000000000, 'currency': 'EUR', 'date': None, 'location': None, 'date_normalized': 'null', 'needs_manual_review': 'false'}, 'status': 'accepted_immediately'}

EXECUTING (Crew): case_010
--- Start Triager ---


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- Routing to Extractor ---


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- Routing to Reviewer ---
{'case_id': 'case_010', 'input': 'Компанія X-Corp (колишня Twitter) уклала угоду з приватною фірмою в ОАЕ.', 'triager_output': {'route': 'news_schema', 'difficulty': 'high'}, 'extractor_output': {'company': 'X-Corp', 'event_type': 'partnership', 'amount': None, 'currency': None, 'date': 'YYYY-MM-DD', 'location': 'ОАЕ'}, 'reviewer_output': {'verdict': 'fallible', 'issues': [{'field': 'date', 'problem': 'The date is not a string or datetime object.'}, {'field': 'location', 'problem': 'The location is not a string.'}]}, 'fallback_triggered': 'false', 'fallback_output': {'date_normalized': 'null', 'needs_manual_review': 'true'}, 'final_output': {'company': 'X-Corp', 'event_type': 'partnership', 'amount': None, 'currency': None, 'date': 'YYYY-MM-DD', 'location': 'ОАЕ', 'date_normalized': 'null', 'needs_manual_review': 'false'}, 'status': 'accepted_immediately'}


# Section 10. Crew logs


In [27]:
import json

def save_to_jsonl(file_path, data):
    with open(file_path, "a", encoding="utf-8") as f:
        f.write(json.dumps(data, ensure_ascii=False) + "\n")

save_to_jsonl("crew_logs_lab13.jsonl", all_results)

# Section 11. Metrics


In [36]:
def calculate_metrics(results, baseline_results=None):

    total = len(results)

    valid = sum(r["status"] in ["accepted_immediately", "accepted_after_repair"] for r in results)

    reviewer_catch = sum(
        r.get("reviewer_output", {}).get("verdict") == "repair_needed"
        for r in results
    )

    fallback = sum(1 for r in results if r.get("fallback_triggered") == "true")

    fallback_success = sum(
        1 for r in results
        if (
            r.get("fallback_triggered")
            and r.get("initial_verdict") == "repair_needed"
            and r.get("reviewer_output", {}).get("verdict") == "accept"
        )
    )

    fallback_total = fallback

    manual = sum(
        r.get("reviewer_output", {}).get("verdict") == "fallback_needed"
        or r.get("status") == "failed"
        for r in results
    )

    metrics = {
        "Valid Final Output Rate": (valid / total) * 100,
        "Reviewer Catch Rate": (reviewer_catch / total) * 100,
        "Fallback Activation Rate": (fallback / total) * 100,
        "Fallback Success Rate": ((fallback_success / fallback_total) * 100 if fallback_total else 0),
        "Manual Review Rate": (manual / total) * 100
    }


    comparison = ""
    if baseline_results:
        baseline_valid = sum(1 for r in baseline_results if "error" not in r)
        baseline_rate = (baseline_valid / len(baseline_results)) * 100
        improvement = metrics["Valid Final Output Rate"] - baseline_rate
        comparison = f"\nSingle-agent Baseline Accuracy: {baseline_rate:.2f}%\nImprovement with Crew: +{improvement:.2f}%"

    print("\n" + "="*40)
    print("FINAL PERFORMANCE METRICS")
    print("="*40)
    for key, value in metrics.items():
        print(f"{key}: {value:.2f}%")

    if comparison:
        print(comparison)
    print("="*40)


In [37]:
calculate_metrics(all_results, single_results)


FINAL PERFORMANCE METRICS
Valid Final Output Rate: 90.00%
Reviewer Catch Rate: 0.00%
Fallback Activation Rate: 0.00%
Fallback Success Rate: 0.00%
Manual Review Rate: 20.00%

Single-agent Baseline Accuracy: 100.00%
Improvement with Crew: +-10.00%


**Single-agent vs crew comparison**

| Rate | Single-agent | Crew |
|------|--------------|------|
|valid output rate|0%|10%|
|consistency error rate|100%|90%|
|hallucinated fields|10%|10%|
|missing required fields|0%|0%|
|cases needing manual review|100%|90%|


# Section 12. Error analysis


**CASE 001**

Помилки:
* Дата 2024-03-15 (у тексті 2024-05-15)
* Reviewer повертає accept, ігноруючи помилку

Категорії помилок: `reviewer missed error`, `final output inconsistent with input`

---

**CASE 002**

Помилки:
* pipeline "впав" через JSON parsing error

Категорії помилок: `invalid JSON`

---

**CASE 003**

Помилки:
* Неправильний event_type: investment (має бути acquisition)
* Неправильний date: "YYYY-MM-DD" (має бути null)
* Reviewer повертає accept, ігноруючи помилку

Категорії помилок:  `reviewer missed error`,  `extractor hallucinated field`

---

**CASE 004**

Помилки:
* Неправильна дата: 2023-10-04 (галлюцинация)

Категорії помилок: `extractor hallucinated field`, `reviewer missed error`

---

**CASE 005**

Помилки:
* Reviewer повертає fallback_needed, але fallback_triggered: false

Категорії помилок: `fallback not triggered`

---

**CASE 006**

Помилки:
* company: "Googlr" (не нормализовано, має бути "Google")

Категорії помилок: `final output inconsistent with input`, `extractor missing normalization`

---

**CASE 007**

Помилки:
* company: "не розташовано" (має бути null)
* Reviewer повертає accept, ігноруючи помилку

Категорії помилок: `extractor hallucinated field`, `reviewer missed error`

---

**CASE 008**

Помилки:
* amount: 0.055 (не правильна нормализація)
* currency: percent
* Reviewer повертає accept, ігноруючи помилку

Категорії помилок: `extractor hallucinated field`, `wrong task interpretation`, `reviewer missed error`

---

**CASE 009**

Помилки:
* Відсутні (хороший кейс)

---

**CASE 010**

Помилки:
* reviewer verdict: "fallible" (нема такої опції)
* Неправильний date: "YYYY-MM-DD" (має бути null)
* Reviewer повертає accept, ігноруючи помилку

Категорії помилок: `wrong reviewer output`, `fallback not triggered`



# Section 13. Generate docs/audit_summary_lab13.md

In [ ]:
report = """# Audit Summary — Lab 13 (Multi-agent crew)

## Який extraction-кейс
**Кейс:** витяг структурованої інформації про економічні та бізнес-події з новинних текстів.

Потрібно отримати:
* `company` (організація/компанія)
* `event_type` (тип події)
* `amount` (сума угоди/інвестиції)
* `currency` (валюта)
* `date` (дата події)
* `location` (локація)

## Які агенти реалізовано
Реалізовано multi-agent pipeline:
1. Triager: визначає тип задачі, задає schema routing
2. Extractor: витягує дані відповідно до schema
3. Reviewer: перевіряє коректність extraction, виявляє помилки та галюцинації
4. Fallback (Repair loop): виправляє помилки після Reviewer

## Скільки test cases
Набір test cases складається з 10 прикладів, кожен з яких має JSON-розмітку з полями `case_id`, `description`, `input` та `expected_behavior`. Обрані кейси покривають коректні новини, відсутні поля, неявні дати, шум (відсотки, аналітика), помилки в тексті (опечатки).

## Valid final output rate
Valid Final Output Rate: 10.00%

## Reviewer catch rate
Reviewer Catch Rate: 0.00%

## Fallback activation rate
Fallback Activation Rate: 0.00%

## Fallback success rate
Fallback Success Rate: 0.00%

## Manual review rate
Manual Review Rate: 90.00%

## Single-agent vs crew comparison
| Rate | Single-agent | Crew |
|------|--------------|------|
|valid output rate|0%|10%|
|consistency error rate|100%|90%|
|hallucinated fields|10%|10%|
|missing required fields|0%|0%|
|cases needing manual review|100%|90%|

## Найкращі приклади роботи crew
**Case 009**
* Правильно визначено company, amount, currency
* Коректна нормалізація (1 млрд -> 1000000000)

Простий, явний кейс без неоднозначностей

**Case 006** — Googlr (з шумом)
* Витягнуто суму: 500000000
* Валюта нормалізована -> USD
* location -> Париж

Попри помилку в назві, extraction загалом адекватний

## Проблемні кейси
**CASE 001**

Помилки:
* Дата 2024-03-15 (у тексті 2024-05-15)
* Reviewer повертає accept, ігноруючи помилку

**CASE 003**

Помилки:
* Неправильний event_type: investment (має бути acquisition)
* Неправильний date: "YYYY-MM-DD" (має бути null)
* Reviewer повертає accept, ігноруючи помилку

**CASE 008**

Помилки:
* amount: 0.055 (неправильна нормализація)
* currency: percent
* Reviewer повертає accept, ігноруючи помилку

## Що варто покращити
1. Посилити Reviewer
2. Заборонити placeholder значення

Never output: "YYYY-MM-DD", "unknown", "не розташовано"

3. Додати перевірку типів полів та enum для event_type
4. Покращити правила нормалізації та виправлення помилок у полях
"""

with open("audit_summary_lab13.md", "w", encoding="utf-8") as f:
    f.write(report)